In [ ]:
! pip install torch torchvision matplotlib
! pip install yolov5

In [ ]:
! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [5]:
print(torch.cuda.is_available())  # Trả về True nếu GPU có thể sử dụng
print(torch.version.cuda) 

False
None


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import torch

import os

class FashionImageCropper:
    def __init__(self, model_path, confidence_threshold=0.5, device='cpu'):
        self.confidence_threshold = confidence_threshold
        self.device = device
        self.model = self.load_model(model_path)

    def load_model(self, model_path):
        model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=True)
        model.conf = self.confidence_threshold
        model.to(self.device)
        return model

    def crop_image(self, input_image, margin=0.1):
        """
        Cắt ảnh lấy trọng tâm sản phẩm thời trang.

        Args:
            input_image (str): Đường dẫn ảnh cần xử lý.
            margin (float): Tỉ lệ mở rộng vùng xung quanh sản phẩm (tính theo chiều rộng và cao).
        
        Returns:
            cropped_image (PIL.Image): Ảnh đã cắt.
        """
        # Đọc ảnh
        img = cv2.imread(input_image)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Chạy mô hình YOLO để phát hiện sản phẩm
        results = self.model(img_rgb)
        detections = results.xyxy[0].cpu().numpy()

        # Lọc bounding boxes có độ tin cậy cao hơn threshold
        detections = [det for det in detections if det[4] >= self.confidence_threshold]

        if not detections:
            raise ValueError("Không phát hiện thấy sản phẩm thời trang trong ảnh.")

        # Chọn bounding box có độ tin cậy cao nhất
        best_box = max(detections, key=lambda x: x[4])
        x_min, y_min, x_max, y_max = best_box[:4]

        # Thêm margin xung quanh bounding box
        img_height, img_width, _ = img.shape
        width_margin = margin * (x_max - x_min)
        height_margin = margin * (y_max - y_min)

        x_min = max(0, int(x_min - width_margin))
        y_min = max(0, int(y_min - height_margin))
        x_max = min(img_width, int(x_max + width_margin))
        y_max = min(img_height, int(y_max + height_margin))

        # Crop ảnh
        cropped_img = img_rgb[int(y_min):int(y_max), int(x_min):int(x_max)]

        # Chuyển về định dạng PIL.Image
        return Image.fromarray(cropped_img)

    def save_cropped_image(self, input_image, output_path, margin=0.1):
        """
        Cắt và lưu ảnh sau khi xử lý.
        Args:
            input_image (str): Đường dẫn ảnh gốc.
            output_path (str): Đường dẫn lưu ảnh đã cắt.
            margin (float): Tỉ lệ mở rộng vùng xung quanh sản phẩm.
        """
        cropped_img = self.crop_image(input_image, margin=margin)
        cropped_img.save(output_path)
        print(f"Ảnh đã được lưu tại: {output_path}")
        
    def crop_all_images_in_folder(self, input_folder, output_folder, margin=0.1):
        """
        Cắt tất cả các ảnh trong thư mục và lưu vào thư mục đầu ra.
        Args:
            input_folder (str): Đường dẫn tới thư mục chứa ảnh cần crop.
            output_folder (str): Đường dẫn tới thư mục lưu ảnh đã crop.
            margin (float): Tỉ lệ mở rộng vùng xung quanh sản phẩm.
        """
        # Kiểm tra nếu thư mục đầu ra không tồn tại thì tạo nó
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
        # Duyệt qua tất cả các ảnh .jpg trong thư mục input_folder
        for filename in os.listdir(input_folder):
            if filename.endswith('.jpg'):
                input_image_path = os.path.join(input_folder, filename)
                output_image_path = os.path.join(output_folder, filename)
                # Cắt ảnh và lưu lại vào thư mục đầu ra
                try:
                    self.save_cropped_image(input_image_path, output_image_path, margin=margin)
                except ValueError:
                    print(f"Không phát hiện sản phẩm thời trang trong ảnh {filename}, bỏ qua.")


In [6]:
# Sử dụng hàm crop_all_images_in_folder
input_folder = 'D:/HK1_2024-2025___NOW/DA_CNTT/New_RS/VibrentDataset/images'  # Đường dẫn tới thư mục chứa ảnh cần crop
output_folder = 'D:/HK1_2024-2025___NOW/DA_CNTT/New_RS/VibrentDataset/images_cropped'  # Đường dẫn tới thư mục lưu ảnh đã crop
model_path = 'yolov5s.pt'  # Đường dẫn tới model YOLOv5 đã huấn luyện

In [ ]:
input_image = "D:/HK1_2024-2025___NOW/DA_CNTT/New_RS/VibrentDataset/sample_images/0a8ef42f26ee43ed9b6482f73a3d7b0e.jpg"  # Đường dẫn ảnh gốc
output_image = "output_image.jpg"  # Đường dẫn ảnh sau xử lý

In [ ]:
# Khởi tạo đối tượng FashionImageCropper
cropper = FashionImageCropper(model_path=model_path, confidence_threshold=0.1)

# Gọi hàm để cắt và lưu tất cả ảnh trong thư mục
cropper.crop_all_images_in_folder(input_folder, output_folder, margin=0.1)

In [ ]:
cropper.crop_image

In [7]:
import os
import cv2
import numpy as np
from PIL import Image
import torch

class SimpleFashionCleaner:
    def __init__(self, model_path, confidence_threshold=0.5, device='cpu'):
        self.confidence_threshold = confidence_threshold
        self.device = device
        self.model = self.load_model(model_path)

    def load_model(self, model_path):
        # Load YOLOv5 model
        model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=True)
        model.conf = self.confidence_threshold
        model.to(self.device)
        return model

    def process_image(self, image_path, output_path, margin=0.1):
        """
        Cắt bỏ phần không phải sản phẩm thời trang.
        
        Args:
            image_path (str): Đường dẫn ảnh gốc.
            output_path (str): Đường dẫn lưu ảnh sau xử lý.
            margin (float): Tỉ lệ mở rộng vùng xung quanh sản phẩm thời trang.
        """
        # Đọc ảnh
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Phát hiện đối tượng với YOLO
        results = self.model(img_rgb)
        detections = results.xyxy[0].cpu().numpy()

        # Lọc các đối tượng có độ tin cậy cao hơn threshold
        detections = [det for det in detections if det[4] >= self.confidence_threshold]

        if not detections:
            raise ValueError("Không phát hiện thấy sản phẩm thời trang trong ảnh.")

        # Tạo mask đen cho ảnh
        mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)

        # Duyệt qua các bounding box và tạo mask
        for det in detections:
            x_min, y_min, x_max, y_max = map(int, det[:4])
            class_id = int(det[5])  # Lớp phát hiện (sản phẩm thời trang)
            
            # Thêm margin cho bounding box
            width_margin = int((x_max - x_min) * margin)
            height_margin = int((y_max - y_min) * margin)
            x_min = max(0, x_min - width_margin)
            y_min = max(0, y_min - height_margin)
            x_max = min(img_rgb.shape[1], x_max + width_margin)
            y_max = min(img_rgb.shape[0], y_max + height_margin)

            # Vẽ mask (vùng sản phẩm là màu trắng)
            cv2.rectangle(mask, (x_min, y_min), (x_max, y_max), 255, thickness=-1)

        # Áp dụng mask để giữ lại vùng sản phẩm thời trang
        result = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

        # Lưu ảnh đã xử lý
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, result_bgr)
        print(f"Ảnh đã lưu tại: {output_path}")

# Sử dụng class
cleaner = SimpleFashionCleaner(model_path="yolov5s.pt", confidence_threshold=0.5, device="cpu")

input_image = "D:/HK1_2024-2025___NOW/DA_CNTT/New_RS/VibrentDataset/sample_images/0a8ef42f26ee43ed9b6482f73a3d7b0e.jpg"  # Đường dẫn ảnh gốc
output_image = "output_image.jpg"  # Đường dẫn ảnh sau xử lý
cleaner.process_image(input_image, output_image, margin=0.1)


Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\ADMIN/.cache\torch\hub\master.zip
YOLOv5  2025-1-20 Python-3.12.0 torch-2.5.1+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
C:\Users\ADMIN/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:894: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Ảnh đã lưu tại: output_image.jpg
